In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm import tqdm

In [12]:
def analyze_protein_cv(protein_file, labels_file, normalization_type, min_locations=3):
    protein_data = pd.read_excel(protein_file, engine='openpyxl')
    labels_data = pd.read_excel(labels_file, engine='openpyxl')
    
    unique_patient_ids = labels_data['patient number'].unique()
    
    batch_columns = [col for col in protein_data.columns if col.startswith(normalization_type)]
    lysis_columns = [col for col in batch_columns if col.endswith('L')]
    
    patient_lysis_columns = {}
    for patient in unique_patient_ids:
        patient_cols = [col for col in lysis_columns if str(patient) in col]
        if len(patient_cols) >= min_locations:
            patient_lysis_columns[patient] = patient_cols
    
    results = []
    
    for index in tqdm(range(len(protein_data)), desc=f'Analyzing {normalization_type} Proteins'):
        row = protein_data.iloc[index]
        protein_identifiers = row[protein_data.columns[:4]]
        
        patient_cvs = []
        for patient, patient_cols in patient_lysis_columns.items():
            measurements = row[patient_cols]
            measurements = measurements[measurements > 0]
            
            if len(measurements) >= min_locations:
                cv = np.std(measurements) / np.mean(measurements) * 100
                patient_cvs.append(cv)
        
        if patient_cvs:
            avg_cv = np.mean(patient_cvs)
            results.append({
                **dict(zip(protein_data.columns[:4], protein_identifiers)),
                'Average CV': avg_cv,
                'Patient CVs': patient_cvs
            })
    
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('Average CV')
    
    return results_df

In [13]:
def plot_cv_distribution(results_df, normalization_type, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    top_20_proteins = results_df.head(20)
    
    for idx, row in tqdm(top_20_proteins.iterrows(), total=len(top_20_proteins), desc=f'Plotting {normalization_type} CV Distributions'):
        plt.figure(figsize=(10, 5))
        plt.hist(row['Patient CVs'], bins=10, edgecolor='black')
        plt.title(f'{normalization_type} Protein CV Distribution\n{" | ".join(row[:4])}')
        plt.xlabel('Coefficient of Variation (%)')
        plt.ylabel('Frequency')
        
        plot_filename = os.path.join(output_dir, f'{normalization_type}_protein_{idx}_cv_dist.png')
        plt.savefig(plot_filename)
        plt.close()
    
    return top_20_proteins

In [14]:
protein_file = '2projects combined-proteinGroups-genes.xlsx'
labels_file = '2projects combined labels.xlsx'

for norm_type in ['Intensity', 'iBAQ', 'LFQ']:
    print(f"Analyzing {norm_type} batch...")
    results = analyze_protein_cv(protein_file, labels_file, norm_type)

    output_dir = f'{norm_type}_cv_plots'
    top_20_proteins = plot_cv_distribution(results, norm_type, output_dir)

    top_20_proteins.to_csv(f'{norm_type}_top_20_proteins.csv', index=False)
    print(f"Analysis for {norm_type} complete.")

Analyzing Intensity batch...


Plotting Intensity CV Distributions: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:02<00:00,  9.49it/s]


Analysis for Intensity complete.
Analyzing iBAQ batch...


Plotting iBAQ CV Distributions: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:02<00:00,  9.79it/s]


Analysis for iBAQ complete.
Analyzing LFQ batch...


Plotting LFQ CV Distributions: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:02<00:00,  9.26it/s]

Analysis for LFQ complete.
